In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(497128, 31)


In [2]:
retailer_features = (
    df.groupby("customerId")
    .agg(
        total_orders=(
            "orderNumber",
            "nunique"
        ),

        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_products=(
            "skuNumber",
            "nunique"
        ),

        last_order=(
            "createdAt",
            "max"
        ),

        first_order=(
            "createdAt",
            "min"
        ),

        shop_type=(
            "shopType",
            "first"
        ),

        retailer_type=(
            "retailerType",
            "first"
        ),

        hub_name=(
            "hubName",
            "first"
        )
    )
    .reset_index()
)

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name
0,USR-100,13,41,15,2026-05-27,2026-01-02,Paan A,HVLF,Crossline Events (Noida)
1,USR-1000,3,7,5,2026-05-16,2026-01-10,General B,HVLF,Instant Foods(Noida)
2,USR-100021,4,31,3,2026-03-30,2026-02-14,General A,HVLF,Instant Foods (SED)
3,USR-100025,2,12,2,2026-02-24,2026-02-18,General A,HVLF,Instant Foods (SED)
4,USR-100062,11,35,12,2026-05-29,2026-02-09,General A,HVLF,Instant Foods (SED)


In [3]:
TODAY = df["createdAt"].max()

retailer_features[
    "days_since_last_order"
] = (
    TODAY -
    retailer_features["last_order"]
).dt.days

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order
0,USR-100,13,41,15,2026-05-27,2026-01-02,Paan A,HVLF,Crossline Events (Noida),4
1,USR-1000,3,7,5,2026-05-16,2026-01-10,General B,HVLF,Instant Foods(Noida),15
2,USR-100021,4,31,3,2026-03-30,2026-02-14,General A,HVLF,Instant Foods (SED),62
3,USR-100025,2,12,2,2026-02-24,2026-02-18,General A,HVLF,Instant Foods (SED),96
4,USR-100062,11,35,12,2026-05-29,2026-02-09,General A,HVLF,Instant Foods (SED),2


In [4]:
fav_category = (
    df.groupby("customerId")
    ["category"]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
    .reset_index()
)

fav_category.columns = [
    "customerId",
    "favorite_category"
]

fav_category.head()

,customerId,favorite_category
0,USR-100,Pan Masala
1,USR-1000,Mouth Fresheners
2,USR-100021,Pan Masala
3,USR-100025,Pan Masala
4,USR-100062,Zarda


In [5]:
retailer_features = (
    retailer_features.merge(
        fav_category,
        on="customerId",
        how="left"
    )
)

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category
0,USR-100,13,41,15,2026-05-27,2026-01-02,Paan A,HVLF,Crossline Events (Noida),4,Pan Masala
1,USR-1000,3,7,5,2026-05-16,2026-01-10,General B,HVLF,Instant Foods(Noida),15,Mouth Fresheners
2,USR-100021,4,31,3,2026-03-30,2026-02-14,General A,HVLF,Instant Foods (SED),62,Pan Masala
3,USR-100025,2,12,2,2026-02-24,2026-02-18,General A,HVLF,Instant Foods (SED),96,Pan Masala
4,USR-100062,11,35,12,2026-05-29,2026-02-09,General A,HVLF,Instant Foods (SED),2,Zarda


In [6]:
fav_brand = (
    df.groupby("customerId")
    ["brand"]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
    .reset_index()
)

fav_brand.columns = [
    "customerId",
    "favorite_brand"
]

In [7]:
retailer_features = (
    retailer_features.merge(
        fav_brand,
        on="customerId",
        how="left"
    )
)

retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category,favorite_brand
0,USR-100,13,41,15,2026-05-27,2026-01-02,Paan A,HVLF,Crossline Events (Noida),4,Pan Masala,DS Group
1,USR-1000,3,7,5,2026-05-16,2026-01-10,General B,HVLF,Instant Foods(Noida),15,Mouth Fresheners,DS Group
2,USR-100021,4,31,3,2026-03-30,2026-02-14,General A,HVLF,Instant Foods (SED),62,Pan Masala,Rajnigandha
3,USR-100025,2,12,2,2026-02-24,2026-02-18,General A,HVLF,Instant Foods (SED),96,Pan Masala,Rajnigandha
4,USR-100062,11,35,12,2026-05-29,2026-02-09,General A,HVLF,Instant Foods (SED),2,Zarda,Tulsi


In [8]:
retailer_features[
    "spend_segment"
] = pd.qcut(
    retailer_features[
        "total_qty"
    ],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

retailer_features[
    [
        "customerId",
        "total_qty",
        "spend_segment"
    ]
].head()

,customerId,total_qty,spend_segment
0,USR-100,41,Medium
1,USR-1000,7,Low
2,USR-100021,31,Medium
3,USR-100025,12,Low
4,USR-100062,35,Medium


In [9]:
retailer_features.to_parquet(
    "../data/processed/retailer_features.parquet",
    index=False
)

print("Saved")

Saved


In [10]:
retailer_features[
    "spend_segment"
].value_counts()

spend_segment
Low       2946
High      2878
Medium    2816
Name: count, dtype: int64

In [11]:
retailer_features.head()

,customerId,total_orders,total_qty,unique_products,last_order,first_order,shop_type,retailer_type,hub_name,days_since_last_order,favorite_category,favorite_brand,spend_segment
0,USR-100,13,41,15,2026-05-27,2026-01-02,Paan A,HVLF,Crossline Events (Noida),4,Pan Masala,DS Group,Medium
1,USR-1000,3,7,5,2026-05-16,2026-01-10,General B,HVLF,Instant Foods(Noida),15,Mouth Fresheners,DS Group,Low
2,USR-100021,4,31,3,2026-03-30,2026-02-14,General A,HVLF,Instant Foods (SED),62,Pan Masala,Rajnigandha,Medium
3,USR-100025,2,12,2,2026-02-24,2026-02-18,General A,HVLF,Instant Foods (SED),96,Pan Masala,Rajnigandha,Low
4,USR-100062,11,35,12,2026-05-29,2026-02-09,General A,HVLF,Instant Foods (SED),2,Zarda,Tulsi,Medium


In [12]:
retailer_features["spend_segment"].value_counts()

spend_segment
Low       2946
High      2878
Medium    2816
Name: count, dtype: int64

In [13]:
retailer_features.shape

(8640, 13)

In [14]:
import pandas as pd

retailer_features = pd.read_parquet(
    "../data/processed/retailer_features.parquet"
)

print(retailer_features.shape)

(8640, 13)


In [15]:
from pathlib import Path

print(Path("../data/processed/retailer_features.parquet").resolve())

C:\Users\Rishit\Desktop\O2R-recommender\data\processed\retailer_features.parquet


In [16]:
print(retailer_features.shape)
print(retailer_features["customerId"].nunique())

(8640, 13)
8640
